<a href="https://colab.research.google.com/github/Lyra0710/Multimodal-Deep-Learning-for-Reliable-GRACE-Water-Storage-Trends/blob/main/notebooks/GRACE_GRACE_FO_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!podaac-data-downloader --help

usage: PO.DAAC bulk-data downloader [-h] -c COLLECTION -d OUTPUTDIRECTORY
                                    [--cycle SEARCH_CYCLES] [-sd STARTDATE]
                                    [-ed ENDDATE] [-f] [-b BBOX] [-dc]
                                    [-dydoy] [-dymd] [-dy] [--offset OFFSET]
                                    [-e EXTENSIONS] [-gr GRANULENAME]
                                    [--process PROCESS_CMD] [--version]
                                    [--verbose] [-p PROVIDER] [--limit LIMIT]
                                    [--dry-run] [--subset]

options:
  -h, --help            show this help message and exit
  -c, --collection-shortname COLLECTION
                        The collection shortname for which you want to
                        retrieve data.
  -d, --data-dir OUTPUTDIRECTORY
                        The directory where data products will be downloaded.
  --cycle SEARCH_CYCLES
                        Cycle number for determining downloads. can be
 

In [2]:
!pip install -q podaac-data-subscriber

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
DATA_DIR = Path(
    "/content/drive/Shareddrives/DATA 255 Group/"
    "255 Group/Group Project/Intermediate Progress/GRACE"
)

DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Folder:", DATA_DIR)
print("Existing files:", list(DATA_DIR.iterdir()))

In [24]:
from pathlib import Path
from getpass import getpass
import pandas as pd
import xarray as xr
import numpy as np

In [7]:
username = input("Earthdata username: ")
password = getpass("Earthdata password: ")

netrc_path = Path.home() / ".netrc"

netrc_path.write_text(
    f"machine urs.earthdata.nasa.gov\n"
    f"login {username}\n"
    f"password {password}\n"
)

netrc_path.chmod(0o600)

del password
print("Earthdata credentials configured.")

Earthdata username: ananyayallapragada@gmail.com
Earthdata password: ··········
Earthdata credentials configured.


In [19]:
!podaac-data-downloader \
    -c TELLUS_GRAC-GRFO_MASCON_CRI_GRID_RL06.3_V4 \
    -d "{DATA_DIR}" \
    --start-date 2019-01-01T00:00:00Z \
    --end-date 2025-12-31T23:59:59Z \
    -e .nc

[2026-10-25 00:00:32,550] {podaac_data_downloader.py:317} INFO - Found 1 total files to download
[2026-10-25 00:00:35,534] {podaac_data_downloader.py:358} INFO - 2026-10-25 00:00:35.534632 SUCCESS: https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/TELLUS_GRAC-GRFO_MASCON_CRI_GRID_RL06.3_V4/GRCTellus.JPL.200204_202607.GLO.RL06.3M.MSCNv04CRI.nc
[2026-10-25 00:00:35,534] {podaac_data_downloader.py:369} INFO - Downloaded Files: 1
[2026-10-25 00:00:35,535] {podaac_data_downloader.py:370} INFO - Failed Files:     0
[2026-10-25 00:00:35,535] {podaac_data_downloader.py:371} INFO - Skipped Files:    0
[2026-10-25 00:00:35,535] {podaac_data_downloader.py:177} INFO - Success Count: 1
[2026-10-25 00:00:35,625] {podaac_data_downloader.py:186} INFO - END




In [20]:
files = sorted(DATA_DIR.glob("*.nc"))

for file in files:
    print(file.name, f"{file.stat().st_size / 1e6:.2f} MB")

print("Total files:", len(files))

GRCTellus.JPL.200204_202607.GLO.RL06.3M.MSCNv04CRI.nc 45.99 MB
Total files: 1


Extracting

In [21]:

# Open the downloaded NetCDF file
ds = xr.open_dataset(files[0])

# Select our study period
grace = ds.sel(time=slice("2019-01-01", "2025-12-31"))

print(grace)

<xarray.Dataset> Size: 529MB
Dimensions:        (time: 84, lat: 360, lon: 720, bounds: 2)
Coordinates:
  * time           (time) datetime64[ns] 672B 2019-01-16T12:00:00 ... 2025-12...
  * lat            (lat) float64 3kB -89.75 -89.25 -88.75 ... 88.75 89.25 89.75
  * lon            (lon) float64 6kB 0.25 0.75 1.25 1.75 ... 358.8 359.2 359.8
Dimensions without coordinates: bounds
Data variables:
    lwe_thickness  (time, lat, lon) float64 174MB ...
    uncertainty    (time, lat, lon) float64 174MB ...
    lat_bounds     (lat, bounds) float64 6kB ...
    lon_bounds     (lon, bounds) float64 12kB ...
    time_bounds    (time, bounds) datetime64[ns] 1kB ...
    land_mask      (lat, lon) float64 2MB ...
    scale_factor   (lat, lon) float64 2MB ...
    mascon_ID      (lat, lon) float64 2MB ...
    GAD            (time, lat, lon) float64 174MB ...
Attributes: (12/53)
    Conventions:                   CF-1.6, ACDD-1.3, ISO 8601
    Metadata_Conventions:          Unidata Dataset Discovery v1.

In [22]:

expected = pd.period_range("2019-01", "2025-12", freq="M")
observed = pd.DatetimeIndex(grace.time.values).to_period("M")

missing = expected.difference(observed)

print("Expected months:", len(expected))
print("Available months:", len(observed))
print("Missing months:", len(missing))
print("Missing month list:", missing.astype(str).tolist())

Expected months: 84
Available months: 84
Missing months: 0
Missing month list: []


Inspecting the land mask

In [23]:


print("Land mask attributes:")
print(grace["land_mask"].attrs)

print("\nUnique land mask values:")
print(np.unique(grace["land_mask"].values))

Land mask attributes:
{'units': 'binary', 'long_name': 'Land_Mask', 'standard_name': 'Land_Mask', 'description': 'Land Mask that was used with the CRI filter'}

Unique land mask values:
[0. 1.]


## Preprocessing

**Since our project focuses on terrestrial water storage, we will have a variable to retain land cells as well**

In [25]:
# Select terrestrial water-storage anomalies
twsa = grace["lwe_thickness"]

# Apply NASA's land mask
twsa_land = twsa.where(grace["land_mask"] == 1)

print(twsa_land)

<xarray.DataArray 'lwe_thickness' (time: 84, lat: 360, lon: 720)> Size: 174MB
array([[[10.19731986, 10.19731986, 10.19731986, ..., 10.19731986,
         10.19731986, 10.19731986],
        [10.19731986, 10.19731986, 10.19731986, ..., 10.19731986,
         10.19731986, 10.19731986],
        [10.19731986, 10.19731986, 10.19731986, ..., 10.19731986,
         10.19731986, 10.19731986],
        ...,
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan]],

       [[10.27972045, 10.27972045, 10.27972045, ..., 10.27972045,
         10.27972045, 10.27972045],
        [10.27972045, 10.27972045, 10.27972045, ..., 10.27972045,
         10.27972045, 10.27972045],
        [10.27972045, 10.27972045, 10.27972045, ..., 10.27972045,
         10.27972045, 10.

1. Examine missing values

In [27]:
twsa = grace["lwe_thickness"]
land = grace["land_mask"] == 1

# Count missing values ONLY over land
missing_land = (twsa.isnull() & land).sum()

# Count valid values ONLY over land
valid_land = (twsa.notnull() & land).sum()

print("Actual missing land observations:", int(missing_land.item()))
print("Valid land observations:", int(valid_land.item()))

Actual missing land observations: 0
Valid land observations: 7216608


# EDA